# Pamoka 09 - Metakognicijos dizaino šablonas


## Nustatymas

Šis užrašų blokas demonstruoja metakognicijos (Metacognition) dizaino šabloną, naudojant Microsoft Agent Framework.

**Išankstiniai reikalavimai:**
- Azure OpenAI diegimas sukonfigūruotas per aplinkos kintamuosius
- Azure CLI autentifikuotas (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kas yra metakognicija?

Metakognicija yra **mąstymas apie mąstymą**. Dirbtinio intelekto agentų kontekste tai reiškia kurti agentus, kurie gali:

- **Savireflektuoti** apie savo išvestis ir samprotavimo procesą
- **Aptikti klaidas** ir deramai atsigauti, užuot tyliai žlugus
- **Įvertinti**, ar jų atsakymai yra išsamūs ir naudingi
- **Prisitaikyti** ir pakeisti strategiją, kai pirminis požiūris nepasiteisina (pvz., grįžti prie atsarginės sistemos)

Metakognityvus agentas ne tik atsako į klausimus — jis stebi savo veiklą ir prisitaiko realiuoju laiku.


## Pagrindiniai ir atsarginiai įrankiai

Įprastas metakognicijos modelis yra **atsarginė strategija**. Agentas pirmiausia bando pagrindinį įrankį; jei jis nepavyksta (pvz., 404 klaida), agentas atpažįsta gedimą ir skaidriai pereina prie atsarginio įrankio.

Tai atspindi realaus pasaulio sistemas, kuriose pagrindinės paslaugos gali būti neprieinamos ir agentai turi patys diagnozuoti problemą prieš pasirinkdami alternatyvų kelią.

Žemiau apibrėžiame du skrydžių paieškos įrankius:
- **Primary** — apima Paryžių, Tokiją ir Barseloną
- **Backup** — apima Berlyną, Sidnėjų ir Niujorką


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Savęs reflektuojantis agentas su klaidų atkūrimu

Žemiau esantis agentas turi pirmiausia pabandyti pagrindinę skrydžio sistemą, atpažinti gedimus ir skaidriai pereiti prie atsarginės sistemos. Po kiekvieno atsakymo jis trumpai įsivertina, ar visiškai atsakė į vartotojo klausimą.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Savęs vertinimo šablonas

Metakognicijos dar viena pusė yra **savęs vertinimas**: atskiras agentas (arba tas pats agentas antrą kartą peržiūrėdamas) peržiūri atsakymą dėl išsamumo, tikslumo ir naudingumo.

Žemiau mes sukuriame `ResponseEvaluator` agentą, kuris įvertina kelionių agento atsakymus trijose dimensijose.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Santrauka

Šiame pamokyme sužinojote, kaip kurti **metakognityvinius agentus** naudojant Microsoft Agent Framework:

- **Savirefleksija**: Agentai, kurie stebi savo sprendimo procesą ir skaidriai praneša, kas įvyko.
- **Klaidų atkūrimas su atsarginėmis priemonėmis**: Pagrindinio + atsarginio įrankio modelis, kai agentas aptinka gedimus (pvz., 404 klaidas) ir automatiškai bando alternatyvų šaltinį.
- **Saviįvertinimas**: Atskiras vertinimo agentas, kuris įvertina atsakymus pagal pilnumą, tikslumą ir naudingumą.

Šie modeliai daro agentus tvirtesnius, skaidresnius ir patikimesnius — kritiškai svarbios savybės gamybiniam diegimui.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Atsakomybės apribojimas:
Šis dokumentas buvo išverstas pasitelkus dirbtinio intelekto vertimo paslaugą Co-op Translator (https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, atkreipkite dėmesį, kad automatizuoti vertimai gali turėti klaidų arba netikslumų. Originalus dokumentas pradinėje kalboje turėtų būti laikomas autoritetingu šaltiniu. Svarbios informacijos atveju rekomenduojame pasitelkti profesionalų vertėją. Mes neatsakome už jokius nesusipratimus ar neteisingus aiškinimus, kilusius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
